<a href="https://colab.research.google.com/github/sethkipsangmutuba/Data-Visualization/blob/main/Session_A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Computation on NumPy Arrays: Universal Functions

###Seth Kipsang
Up until now, we have been discussing some of the basic nuts and bolts of NumPy. In the next few sections, we will dive into why NumPy is so important in the Python data science ecosystem. At its core, NumPy provides a simple yet powerful interface for efficient computation over arrays of data.

Computation on NumPy arrays can be extremely fast, or unexpectedly slow. The key difference lies in how the operations are performed. To achieve high performance, NumPy relies on vectorized operations, which are implemented through universal functions (ufuncs). These functions are designed to operate on entire arrays at once, rather than requiring explicit loops over individual elements.

This section builds intuition for why ufuncs are necessary and how they significantly improve computational efficiency. It then introduces a range of commonly used arithmetic ufuncs available in NumPy, which form the foundation for fast numerical computation in Python.

In [332]:
import pandas as pd
from sklearn.datasets import load_iris

# Load Iris dataset
iris = load_iris()

# Convert to DataFrame
iris_df = pd.DataFrame(
    data=iris.data,
    columns=iris.feature_names
)

# Add target column (species)
iris_df["species"] = iris.target
iris_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [333]:
iris_df.shape


(150, 5)

In [334]:
iris_df.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'species'],
      dtype='object')

##3.1) The Slowness of Loops

CPython is slow due to dynamic typing and interpretation overhead.

Python cannot fully compile loops into optimized machine code like C/Fortran.

Some solutions exist:

- PyPy (JIT compilation)  
- Cython (Python → C)  
- Numba (LLVM compilation)  

Slow performance appears when repeating many small operations (e.g., loops over arrays).

Example: computing reciprocals using a Python loop is inefficient for large arrays.

Even with simple operations, execution becomes slow at scale.

Main bottleneck: type checking and function dispatch at each loop iteration.

Each step requires Python to determine data type and operation dynamically.

Compiled languages avoid this by knowing types in advance, enabling faster execution.

In [335]:
import numpy as np
import time

# Use Iris dataset

x = iris_df.iloc[:, 0].to_numpy()  # first feature column

# 1. Python loop (element-wise reciprocal)

def reciprocal_python(arr):
    result = []
    for val in arr:
        result.append(1 / val)
    return result

start = time.time()
loop_result = reciprocal_python(x)
loop_time = time.time() - start
loop_time

0.00013494491577148438

In [336]:
# 2. NumPy vectorized computation

start = time.time()
numpy_result = 1 / x
numpy_time = time.time() - start
numpy_time

0.00018525123596191406

In [337]:
# 3. Results comparison

print("Python loop time:", loop_time)
print("NumPy vectorized time:", numpy_time)

print("\nFirst 5 loop results:", loop_result[:5])
print("First 5 NumPy results:", numpy_result[:5])

print("\nMatch check:", np.allclose(loop_result, numpy_result))

Python loop time: 0.00013494491577148438
NumPy vectorized time: 0.00018525123596191406

First 5 loop results: [np.float64(0.19607843137254904), np.float64(0.2040816326530612), np.float64(0.2127659574468085), np.float64(0.2173913043478261), np.float64(0.2)]
First 5 NumPy results: [0.19607843 0.20408163 0.21276596 0.2173913  0.2       ]

Match check: True


##3.2) Introducing UFuncs

NumPy provides vectorized operations, avoiding explicit Python loops.

These operations run in compiled, optimized code layers, making execution faster.

A ufunc (universal function) applies operations element-wise across arrays.

Vectorization replaces loop-based computation with array-wide expressions.

---

### Key idea:

- Loop version is slow  
- Vectorized NumPy operation is fast  

---

### Example insight:

Element-wise reciprocal via loop vs direct array operation shows identical output.

NumPy version executes orders of magnitude faster.

---

### Performance observation:

Large arrays benefit most from vectorization (milliseconds vs seconds).

---

### Ufunc behavior:

Works between:
- array and scalar  
- array and array  

Supports:
- 1D arrays  
- multidimensional arrays  

---

### Example concept:

Element-wise operations like division or exponentiation apply automatically across all elements.

---

Always prefer vectorized NumPy expressions over explicit Python loops for numerical computation.

In [338]:
import numpy as np
import time

# Iris dataset

x = iris_df.iloc[:, 0].to_numpy()


# 1. Explicit Python loop

def reciprocal_loop(arr):
    result = []
    for val in arr:
        result.append(1 / val)
    return result

start = time.time()
loop_result = reciprocal_loop(x)
loop_time = time.time() - start
loop_time

0.0001266002655029297

In [339]:
# =============================
# 2. NumPy ufunc (vectorized operation)
# =============================
start = time.time()
vectorized_result = np.reciprocal(x)
ufunc_time = time.time() - start
ufunc_time

0.0002288818359375

In [340]:
# 3. Scalar–array ufunc behavior

scalar_add = x + 2          # broadcast scalar
scalar_mul = x * 10         # broadcast scalar multiplication

In [341]:
# 4. array–array ufunc behavior

y = iris_df.iloc[:, 1].to_numpy()
pairwise_add = x + y
pairwise_div = x / y

In [342]:
# 5. Output comparison

print("Loop time:", loop_time)
print("UFunc time:", ufunc_time)

print("\nFirst 5 loop results:", loop_result[:5])
print("First 5 ufunc results:", vectorized_result[:5])

print("\nMatch check:", np.allclose(loop_result, vectorized_result))

print("\nScalar broadcasting example (x + 2):", scalar_add[:5])
print("Array–array example (x + y):", pairwise_add[:5])

Loop time: 0.0001266002655029297
UFunc time: 0.0002288818359375

First 5 loop results: [np.float64(0.19607843137254904), np.float64(0.2040816326530612), np.float64(0.2127659574468085), np.float64(0.2173913043478261), np.float64(0.2)]
First 5 ufunc results: [0.19607843 0.20408163 0.21276596 0.2173913  0.2       ]

Match check: True

Scalar broadcasting example (x + 2): [7.1 6.9 6.7 6.6 7. ]
Array–array example (x + y): [8.6 7.9 7.9 7.7 8.6]


##3.3)  Exploring NumPy’s UFuncs

Ufuncs come in two types:

- **Unary ufuncs** → operate on a single input  
- **Binary ufuncs** → operate on two inputs  

---

## Array Arithmetic

NumPy ufuncs integrate with Python arithmetic operators.

Operations like addition, subtraction, multiplication, and division are applied element-wise.

Floor division (`//`) performs integer division per element.

Arithmetic operators are shorthand for ufuncs like:

- `+` → `np.add`  
- `-` → `np.subtract`  
- `*` → `np.multiply`  
- `/` → `np.divide`  
- `**` → `np.power`  
- `%` → `np.mod`  

---

### Key properties

- Expressions follow normal mathematical order of operations  
- Operations can be chained and combined naturally  

---

## Absolute Value

Works on arrays element-wise:

- `abs(x)` or `np.abs(x)`

Handles complex numbers by returning magnitude.

---

## Trigonometric Functions

Apply element-wise to arrays:

- `sin`, `cos`, `tan`

Inverse trig functions also available:

- `arcsin`, `arccos`, `arctan`

Numerical precision may produce very small approximation errors near zero.

---

## Exponentials and Logarithms

### Exponentials:

- `exp`, `exp2`, `power`

### Logarithms:

- `log` (natural log)  
- `log2`, `log10`

### Special stable functions:

- `expm1` → accurate for small values of x  
- `log1p` → accurate for log(1 + x)  

---

## Specialized Ufuncs

NumPy includes advanced mathematical ufuncs:

- hyperbolic functions  
- bitwise operations  
- rounding and comparisons  

Additional specialized functions exist in `scipy.special`:

- gamma functions  
- beta functions  
- error function (`erf`, `erfc`, `erfinv`)  

---

Ufuncs enable fast, vectorized mathematical computation across arrays.

They replace explicit loops and significantly improve performance and clarity.

In [343]:
import numpy as np

# -----------------------------
# Iris dataset
# -----------------------------
x = iris_df.iloc[:, 0].to_numpy()
y = iris_df.iloc[:, 1].to_numpy()

# 1. Unary UFuncs

abs_vals = np.abs(x)

sin_vals = np.sin(x)
cos_vals = np.cos(x)
tan_vals = np.tan(x)

exp_vals = np.exp(x)
log_vals = np.log(x + 1e-8)  # avoid log(0)

# Inverse trig (bounded stability check)
arcsin_vals = np.arcsin(np.clip(x / np.max(x), -1, 1))

print("Unary ufunc example (abs):", abs_vals[:5])
print("Unary ufunc example (sin):", sin_vals[:5])


Unary ufunc example (abs): [5.1 4.9 4.7 4.6 5. ]
Unary ufunc example (sin): [-0.92581468 -0.98245261 -0.99992326 -0.993691   -0.95892427]


In [344]:
# =============================
# 2. Binary UFuncs (element-wise operations)
# =============================
add_vals = x + y
sub_vals = x - y
mul_vals = x * y
div_vals = x / (y + 1e-8)
floor_div = x // (y + 1e-8)
power_vals = x ** 2
mod_vals = x % (y + 1e-8)

print("\nBinary ufunc example (x + y):", add_vals[:5])
print("Binary ufunc example (x * y):", mul_vals[:5])



Binary ufunc example (x + y): [8.6 7.9 7.9 7.7 8.6]
Binary ufunc example (x * y): [17.85 14.7  15.04 14.26 18.  ]


In [345]:
# 3. Function vs Operator equivalence

print("\nnp.add == + :", np.allclose(np.add(x, y), x + y))
print("np.multiply == * :", np.allclose(np.multiply(x, y), x * y))
print("np.power == ** :", np.allclose(np.power(x, 2), x ** 2))



np.add == + : True
np.multiply == * : True
np.power == ** : True


In [346]:
# 4. Special numerical stability functions

expm1_vals = np.expm1(x * 1e-3)
log1p_vals = np.log1p(x * 1e-3)

print("\nexpm1 example:", expm1_vals[:5])
print("log1p example:", log1p_vals[:5])



expm1 example: [0.00511303 0.00491202 0.00471106 0.0046106  0.00501252]
log1p example: [0.00508704 0.00488803 0.00468899 0.00458945 0.00498754]


In [347]:
# 5. Rounding and comparisons

rounded = np.round(x, 2)
greater_than = x > np.mean(x)

print("\nRounded values:", rounded[:5])
print("Boolean mask:", greater_than[:5])


Rounded values: [5.1 4.9 4.7 4.6 5. ]
Boolean mask: [False False False False False]


##3.4) Advanced Ufunc Features

---

## Specifying Output

Ufuncs allow controlling where results are stored using an `out` parameter.

This avoids creation of temporary arrays → improves memory efficiency.

Useful for large-scale computations.

---

### Key idea

Instead of creating new arrays, results can be written directly into existing memory locations.

---

## Using Array Views

Output can be written into subsections of arrays (slices/views).

Example concept: storing results in every other element of an array.

This helps optimize memory usage further.

---

## Performance Insight

Without `out`, NumPy creates temporary arrays before assignment.

This introduces:
- extra memory overhead  
- slower execution for large datasets  

With `out`, computation is done in-place, reducing overhead.

---

## Aggregates (Reduction Operations)

Binary ufuncs support aggregation via `reduce`.

`reduce` repeatedly applies an operation until a single value remains.

---

### Examples of behavior

- Addition reduction → computes sum of all elements  
- Multiplication reduction → computes product of all elements  

---

## Accumulate

`accumulate` stores intermediate results instead of only the final value.

Useful for tracking progressive computation:

- cumulative sum  
- cumulative product  

---

### Note

NumPy also provides optimized alternatives:

- `sum`  
- `product`  
- `cumulative sum`  
- `cumulative product`  

---

## Outer Products

`outer` applies a ufunc to all pairs of elements from two arrays.

Produces a full pairwise operation matrix.

---

### Key idea

Turns two 1D arrays into a 2D interaction table.

---

### Typical use cases

- multiplication tables  
- pairwise comparisons  
- kernel-like computations  

---

## Additional Ufunc Tools

- `at` and `reduceat` support indexed and partial reductions  
- Useful in advanced array manipulation workflows  

---

## Broadcasting (Preview Concept)

Ufuncs can operate on arrays of different shapes and sizes.

This automatic alignment is called **broadcasting**.

It is a core feature enabling NumPy’s flexible computation model.

---

## Tip

Ufunc behavior can be explored interactively using:

- tab completion  
- built-in documentation system (`help`)  

---

Advanced ufunc features enable:

- memory-efficient computation  
- structured aggregation  
- pairwise operations  
- flexible shape-based computation (broadcasting)  

In [348]:
import numpy as np

# Iris dataset

x = iris_df.iloc[:, 0].to_numpy()
y = iris_df.iloc[:, 1].to_numpy()

# 1. Specifying Output (out parameter - memory reuse)

out_array = np.empty_like(x)

np.add(x, y[:x.shape[0]], out=out_array)

print("Out parameter result (first 5):", out_array[:5])


Out parameter result (first 5): [8.6 7.9 7.9 7.7 8.6]


In [349]:
# 2. Writing into array views (slice-aligned operation)

target = np.zeros_like(x)

# Correct: ensure shapes match
np.multiply(x[::2], 2, out=target[::2])

print("\nSlice-based in-place write (first 10):", target[:10])



Slice-based in-place write (first 10): [10.2  0.   9.4  0.  10.   0.   9.2  0.   8.8  0. ]


In [350]:
# 3. Reduction (reduce → collapse to scalar)

sum_all = np.add.reduce(x)
prod_all = np.multiply.reduce(x)

print("\nReduce sum:", sum_all)
print("Reduce product:", prod_all)



Reduce sum: 876.5
Reduce product: 2.2574399991830563e+114


In [351]:
# 4. Accumulation (running computation)

cumsum = np.add.accumulate(x)
cumprod = np.multiply.accumulate(x)

print("\nAccumulate sum (first 5):", cumsum[:5])
print("Accumulate product (first 5):", cumprod[:5])


Accumulate sum (first 5): [ 5.1 10.  14.7 19.3 24.3]
Accumulate product (first 5): [   5.1      24.99    117.453   540.2838 2701.419 ]


In [352]:
# 5. Outer product (pairwise interaction matrix)

outer_matrix = np.multiply.outer(x[:5], y[:5])

print("\nOuter product matrix:\n", outer_matrix)



Outer product matrix:
 [[17.85 15.3  16.32 15.81 18.36]
 [17.15 14.7  15.68 15.19 17.64]
 [16.45 14.1  15.04 14.57 16.92]
 [16.1  13.8  14.72 14.26 16.56]
 [17.5  15.   16.   15.5  18.  ]]


In [353]:
# 6. Indexed operations using at()

test_arr = x.copy()

np.add.at(test_arr, [0, 1, 2], 100)

print("\nAfter np.add.at (first 5):", test_arr[:5])



After np.add.at (first 5): [105.1 104.9 104.7   4.6   5. ]


In [354]:
# 7. Broadcasting preview consistency check

broadcast_example = x + 2

print("\nBroadcasting example (x + 2):", broadcast_example[:5])


Broadcasting example (x + 2): [7.1 6.9 6.7 6.6 7. ]
